# 06 - Cleaning and the Silver layer

**Input:** `bronze.kaggle_listings` (raw data in SQL Server) + the neighbourhood features from notebook 04.
**Output:**
- `data/processed/airbnb_clean.xlsx` (the cleaned dataset as an Excel file, as required by the project)
- `silver.listings` and `silver.neighbourhood_features` in SQL Server

Every cleaning decision below comes from the findings in notebook 02.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.types import NVARCHAR

PROCESSED = Path("../data/processed")

DRIVER = sorted(d for d in pyodbc.drivers() if "ODBC Driver" in d and "SQL Server" in d)[-1]
conn_str = (
    f"DRIVER={{{DRIVER}}};SERVER=.;DATABASE=AirbnbEurope;"
    "Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": conn_str}), fast_executemany=True)

## Step 1: Read the raw data from the Bronze layer
The Silver layer is always built **from Bronze**, never from the original files.

In [2]:
df = pd.read_sql("SELECT * FROM bronze.kaggle_listings", engine)
rows_start = len(df)
print(df.shape)
df.head()

(51707, 24)


,source_row_id,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type,_source,_loaded_at
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,...,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekdays,kaggle,2026-10-24 20:59:07
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,...,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekdays,kaggle,2026-10-24 20:59:07
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,...,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekdays,kaggle,2026-10-24 20:59:07
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,...,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekdays,kaggle,2026-10-24 20:59:07
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,...,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekdays,kaggle,2026-10-24 20:59:07


## Step 2: Remove the load-tracking columns
`_source` and `_loaded_at` describe the Bronze load, not the listings, so they are not needed in Silver.

In [3]:
df = df.drop(columns=["_source", "_loaded_at"])

## Step 3: Check that the room columns are really redundant before removing them
`room_shared` and `room_private` should carry the same information as `room_type`.
We prove it first instead of assuming it.

In [4]:
print(pd.crosstab(df["room_type"], [df["room_shared"], df["room_private"]]))

redundant = (
    (df["room_shared"] == (df["room_type"] == "Shared room")).all()
    and (df["room_private"] == (df["room_type"] == "Private room")).all()
)
print("\nroom_shared / room_private fully explained by room_type:", redundant)

room_shared      False        True 
room_private     False  True  False
room_type                          
Entire home/apt  32648      0     0
Private room         0  18693     0
Shared room          0      0   366

room_shared / room_private fully explained by room_type: True


## Step 4: Drop redundant columns
- `room_shared`, `room_private`: same information as `room_type` (proved above).
- `attr_index`, `rest_index`: the `_norm` versions hold the same information on a 0-100 scale within each city,
  which is easier to compare between cities.

In [5]:
df = df.drop(columns=["room_shared", "room_private", "attr_index", "rest_index"])
print(df.shape)

(51707, 18)


## Step 5: Fix data types
- `person_capacity` is stored as a decimal (2.0, 4.0) but a number of guests is always whole.
- `multi` and `biz` are stored as 0 / 1 numbers while `host_is_superhost` is True / False.
  We make all yes/no columns the same type (boolean).

In [6]:
print("person_capacity values:", sorted(df["person_capacity"].unique()))
print("multi values:", sorted(df["multi"].unique()), "| biz values:", sorted(df["biz"].unique()))

df["person_capacity"] = df["person_capacity"].astype(int)
df["bedrooms"] = df["bedrooms"].astype(int)
df["multi"] = df["multi"].astype(bool)
df["biz"] = df["biz"].astype(bool)
df["host_is_superhost"] = df["host_is_superhost"].astype(bool)

print("\nBoth multi and biz True at the same time:", (df["multi"] & df["biz"]).sum(), "rows")

person_capacity values: [np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]
multi values: [np.int64(0), np.int64(1)] | biz values: [np.int64(0), np.int64(1)]

Both multi and biz True at the same time: 0 rows


## Step 6: Rename columns to clear names
Short names like `biz` or `realSum` are hard to understand in SQL and Power BI.

In [7]:
df = df.rename(columns={
    "realSum": "price_eur",
    "multi": "host_has_2_to_4_listings",
    "biz": "host_has_over_4_listings",
    "guest_satisfaction_overall": "guest_satisfaction",
    "dist": "dist_center_km",
    "metro_dist": "dist_metro_km",
    "attr_index_norm": "attraction_index",
    "rest_index_norm": "restaurant_index",
    "lng": "longitude",
    "lat": "latitude",
})
df.columns.tolist()

['source_row_id',
 'price_eur',
 'room_type',
 'person_capacity',
 'host_is_superhost',
 'host_has_2_to_4_listings',
 'host_has_over_4_listings',
 'cleanliness_rating',
 'guest_satisfaction',
 'bedrooms',
 'dist_center_km',
 'dist_metro_km',
 'attraction_index',
 'restaurant_index',
 'longitude',
 'latitude',
 'city',
 'day_type']

## Step 7: Remove extreme price outliers
Decision from notebook 02: listings priced above **30x their city's median** are implausible
(for example private rooms for 2 people above EUR 12,000) and are removed.
Listings between 20x and 30x are kept because they include genuine large luxury homes.

In [8]:
city_median = df.groupby("city")["price_eur"].transform("median")
is_extreme = df["price_eur"] > 30 * city_median

print("Rows removed:", is_extreme.sum())
display(df.loc[is_extreme, ["city", "room_type", "bedrooms", "person_capacity", "price_eur"]]
        .sort_values("price_eur", ascending=False))

df = df[~is_extreme].copy()
print("Rows left:", len(df))

Rows removed: 18


,city,room_type,bedrooms,person_capacity,price_eur
3590,Athens,Entire home/apt,1,2,18545.450285
34803,Paris,Entire home/apt,1,2,16445.614689
24348,London,Entire home/apt,3,3,15499.894165
48380,Vienna,Private room,1,2,13664.305916
50787,Vienna,Private room,1,2,13656.358834
48215,Vienna,Entire home/apt,1,4,12942.991375
30626,London,Entire home/apt,2,6,12937.275101
31635,London,Private room,1,2,12929.513864
25945,London,Private room,1,2,12886.239093
25629,London,Entire home/apt,2,6,12886.239093


Rows left: 51689


## Step 8: Add the neighbourhood features from notebook 04
`airbnb_enriched.csv` was built from `airbnb_master_raw.csv` with a left merge, which keeps the original row order.
So we can take `Unnamed: 0` from the raw file to rebuild the key `(city, day_type, source_row_id)`
that identifies each listing in both datasets.

In [9]:
raw_ids = pd.read_csv(PROCESSED / "airbnb_master_raw.csv", usecols=["Unnamed: 0"])
enriched = pd.read_csv(PROCESSED / "airbnb_enriched.csv")
assert len(raw_ids) == len(enriched)
enriched["source_row_id"] = raw_ids["Unnamed: 0"].values

nb_cols = ["neighbourhood"] + [c for c in enriched.columns if c.startswith("nb_") and c not in ("nb_match_method", "nb_distance_m")]
key = ["city", "day_type", "source_row_id"]

rows_before = len(df)
df = df.merge(enriched[key + nb_cols], on=key, how="left", validate="one_to_one")
print("Same number of rows after merge:", len(df) == rows_before)
print("Listings without a neighbourhood:", df["neighbourhood"].isna().sum())
print("\nMissing values in neighbourhood features:")
print(df[nb_cols].isna().sum())

Same number of rows after merge: True
Listings without a neighbourhood: 0

Missing values in neighbourhood features:
neighbourhood             0
nb_listings               4
nb_median_reviews         4
nb_avg_rating             9
nb_superhost_share        4
nb_median_amenities       4
nb_median_availability    4
nb_avg_host_listings      4
nb_entire_home_share      4
dtype: int64


## Step 9: Add a listing ID
The Kaggle data has no ID. A simple number from 1 to N becomes the **primary key** in SQL later.

In [10]:
df = df.sort_values(["city", "day_type", "source_row_id"]).reset_index(drop=True)
df.insert(0, "listing_id", np.arange(1, len(df) + 1))
df = df.drop(columns=["source_row_id"])
df.head()

,listing_id,price_eur,room_type,person_capacity,host_is_superhost,host_has_2_to_4_listings,host_has_over_4_listings,cleanliness_rating,guest_satisfaction,bedrooms,...,day_type,neighbourhood,nb_listings,nb_median_reviews,nb_avg_rating,nb_superhost_share,nb_median_amenities,nb_median_availability,nb_avg_host_listings,nb_entire_home_share
0,1,194.033698,Private room,2,False,True,False,10.0,93.0,1,...,weekdays,Noord-West,315.0,7.0,4.873050,0.158730,35.0,12.0,1.641270,0.866667
1,2,344.245776,Private room,4,False,False,False,8.0,85.0,1,...,weekdays,Centrum-West,1234.0,29.0,4.814254,0.301459,29.0,60.0,3.874392,0.619125
2,3,264.101422,Private room,2,False,False,True,9.0,87.0,1,...,weekdays,IJburg - Zeeburgereiland,190.0,11.0,4.867904,0.200000,37.0,23.0,1.978947,0.768421
3,4,433.529398,Private room,4,False,False,True,9.0,90.0,2,...,weekdays,Centrum-West,1234.0,29.0,4.814254,0.301459,29.0,60.0,3.874392,0.619125
4,5,485.552926,Private room,2,True,False,False,10.0,98.0,1,...,weekdays,Centrum-West,1234.0,29.0,4.814254,0.301459,29.0,60.0,3.874392,0.619125


## Step 10: Final checks

In [11]:
print("Rows at start:", rows_start, "| rows now:", len(df), "| removed:", rows_start - len(df))
print("Duplicate rows:", df.drop(columns="listing_id").duplicated().sum())
print("listing_id unique:", df["listing_id"].is_unique)
print("\nMissing values per column (only columns with any):")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "none")
print()
df.info()

Rows at start: 51707 | rows now: 51689 | removed: 18
Duplicate rows: 0
listing_id unique: True

Missing values per column (only columns with any):
nb_listings               4
nb_median_reviews         4
nb_avg_rating             9
nb_superhost_share        4
nb_median_amenities       4
nb_median_availability    4
nb_avg_host_listings      4
nb_entire_home_share      4
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51689 entries, 0 to 51688
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   listing_id                51689 non-null  int64  
 1   price_eur                 51689 non-null  float64
 2   room_type                 51689 non-null  object 
 3   person_capacity           51689 non-null  int64  
 4   host_is_superhost         51689 non-null  bool   
 5   host_has_2_to_4_listings  51689 non-null  bool   
 6   host_has_over_4_listings  51689 non-null  bool   
 7   cleanl

## Step 11: Save the cleaned data as Excel
Project requirement 3: *"Save the final cleaned dataset as an Excel file."*
Writing about 50,000 rows to Excel can take up to a minute.

In [12]:
excel_path = PROCESSED / "airbnb_clean.xlsx"
df.to_excel(excel_path, index=False, sheet_name="listings")
print("Saved:", excel_path, f"({excel_path.stat().st_size / 1e6:.1f} MB)")

Saved: ..\data\processed\airbnb_clean.xlsx (9.1 MB)


## Step 12: Write the Silver layer
Text columns are stored as **NVARCHAR** (Unicode) so Greek and Hungarian neighbourhood names are kept exactly.

In [13]:
neighbourhoods = pd.read_csv(PROCESSED / "neighbourhood_features.csv")


def unicode_text(frame):
    """Map every text column to NVARCHAR(max) so no characters are lost."""
    return {
        col: NVARCHAR(None)
        for col in frame.columns
        if pd.api.types.is_object_dtype(frame[col]) or pd.api.types.is_string_dtype(frame[col])
    }


df.to_sql("listings", engine, schema="silver", if_exists="replace", index=False,
          chunksize=5000, dtype=unicode_text(df))
neighbourhoods.to_sql("neighbourhood_features", engine, schema="silver", if_exists="replace", index=False,
                      chunksize=5000, dtype=unicode_text(neighbourhoods))

with engine.connect() as conn:
    sample = conn.execute(text(
        "SELECT TOP 3 neighbourhood FROM silver.neighbourhood_features WHERE city = 'Athens'"
    )).scalars().all()
print("Athens neighbourhood names in SQL Server:", sample)

checks = {"silver.listings": len(df), "silver.neighbourhood_features": len(neighbourhoods)}
with engine.connect() as conn:
    for table, expected in checks.items():
        in_sql = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"{table:<30} pandas: {expected:>6}   SQL: {in_sql:>6}   match: {expected == in_sql}")

Athens neighbourhood names in SQL Server: ['1Ο ΝΕΚΡΟΤΑΦΕΙΟ', 'ΑΓΙΟΣ ΕΛΕΥΘΕΡΙΟΣ', 'ΑΓΙΟΣ ΚΩΝΣΤΑΝΤΙΝΟΣ-ΠΛΑΤΕΙΑ ΒΑΘΗΣ']
silver.listings                pandas:  51689   SQL:  51689   match: True
silver.neighbourhood_features  pandas:    515   SQL:    515   match: True
